In [1]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
PROJECT_ROOT=Path.cwd().parent
# Khai báo đường dẫn
DATA_DIR=PROJECT_ROOT/"data"/"raw"

PARQUET_PATH=DATA_DIR/"yellow_tripdata_2026-01.parquet"
LOOKUP_PATH=DATA_DIR/"taxi_zone_lookup.csv"

# Kiểm tra file tồn tại
print(DATA_DIR)
print("Project root: ",PROJECT_ROOT)
print("Parquet exists: ",PARQUET_PATH.exists())
print("Lookup exists: ",LOOKUP_PATH.exists())

c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project\data\raw
Project root:  c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project
Parquet exists:  True
Lookup exists:  True


In [3]:
df=pd.read_parquet(PARQUET_PATH)
zones=pd.read_csv(LOOKUP_PATH)

display(df.head(10))
display(zones.head(10))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0.0,2.90,1.0,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [4]:
# Kiểm tra giá trị null
showDFnull=df.isnull().sum()
showZNnull=zones.isnull().sum()
print(f"Giá trị null \n{showDFnull[showDFnull>0]}\n")
print(f"Giá trị Zones null \n{showZNnull[showZNnull>0]}")


Giá trị null 
passenger_count         1088058
RatecodeID              1088058
store_and_fwd_flag      1088058
congestion_surcharge    1088058
Airport_fee             1088058
dtype: int64

Giá trị Zones null 
Borough         1
Zone            1
service_zone    2
dtype: int64


In [5]:

# Kiểm tra NULL trong bảng tra cứu khu vực
zones_null = zones[zones.isnull().any(axis=1)]

print("Số dòng có NULL trong bảng zones:", len(zones_null))
display(zones_null)


Số dòng có NULL trong bảng zones: 2


,LocationID,Borough,Zone,service_zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


In [6]:
colNull= [
    'passenger_count',
    'RatecodeID',
    'store_and_fwd_flag',
    'congestion_surcharge',
    'Airport_fee'
]
null_per_row = df[colNull].isnull().sum(axis=1)
# df[colNull].isnull().sum(axis=1)
null_per_row


0          0
1          0
2          0
3          0
4          0
          ..
3724884    5
3724885    5
3724886    5
3724887    5
3724888    5
Length: 3724889, dtype: int64

In [7]:
df.loc[null_per_row>0]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2636831,2,2026-01-01 00:28:47,2026-01-01 00:49:30,NaN,10.58,NaN,None,43,200,0,82.01,0.0,0.5,0.0,3.18,1.0,89.19,NaN,NaN,0.00
2636832,2,2026-01-01 00:54:24,2026-01-01 01:15:01,NaN,11.04,NaN,None,200,51,0,51.77,0.0,0.5,0.0,0.00,1.0,53.27,NaN,NaN,0.00
2636833,2,2026-01-01 00:12:31,2026-01-01 00:41:02,NaN,4.25,NaN,None,142,234,0,32.17,0.0,0.5,0.0,0.00,1.0,36.92,NaN,NaN,0.75
2636834,2,2026-01-01 00:48:56,2026-01-01 01:08:00,NaN,2.05,NaN,None,90,114,0,18.29,0.0,0.5,1.0,0.00,1.0,24.04,NaN,NaN,0.75
2636835,2,2026-01-01 00:32:09,2026-01-01 00:41:05,NaN,0.97,NaN,None,237,163,0,15.02,0.0,0.5,0.0,0.00,1.0,22.77,NaN,NaN,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,None,237,161,0,17.09,0.0,0.5,0.0,0.00,1.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,None,42,42,0,23.19,0.0,0.5,0.0,0.00,1.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,None,137,69,0,29.21,0.0,0.5,0.0,0.00,1.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,None,137,162,0,19.19,0.0,0.5,0.0,0.00,1.0,23.94,NaN,NaN,0.75


In [8]:
# Kiểm tra các cột chứa các giá trị null
# passenger_count         1088058
# RatecodeID              1088058
# store_and_fwd_flag      1088058
# congestion_surcharge    1088058
# Airport_fee             1088058
df_null_group= df.loc[null_per_row>0]

display(df_null_group[
        [
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "fare_amount",
            "total_amount"
        ]
    ].head(10))


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount
2636831,2,2026-01-01 00:28:47,2026-01-01 00:49:30,10.58,82.01,89.19
2636832,2,2026-01-01 00:54:24,2026-01-01 01:15:01,11.04,51.77,53.27
2636833,2,2026-01-01 00:12:31,2026-01-01 00:41:02,4.25,32.17,36.92
2636834,2,2026-01-01 00:48:56,2026-01-01 01:08:00,2.05,18.29,24.04
2636835,2,2026-01-01 00:32:09,2026-01-01 00:41:05,0.97,15.02,22.77
2636836,2,2026-01-01 00:24:17,2026-01-01 00:31:51,1.99,13.87,25.37
2636837,2,2026-01-01 00:49:31,2026-01-01 01:21:03,6.44,24.77,29.52
2636838,2,2026-01-01 00:39:35,2026-01-01 01:27:01,7.40,59.36,76.93
2636839,2,2026-01-01 00:27:04,2026-01-01 00:36:03,0.98,9.87,14.62
2636840,2,2026-01-01 00:44:00,2026-01-01 01:08:00,4.30,33.59,44.09


In [9]:
df_result=df_null_group.groupby('VendorID')['trip_distance'].count().reset_index(name='count')
display(df_result)

,VendorID,count
0,1,139406
1,2,944635
2,6,4017


In [10]:
# tạo 2 biến null và ko null để so sánh các chuyến đi
df_null_group = df.loc[null_per_row == 5]
df_complete_group = df.loc[null_per_row == 0]


In [11]:
# So sánh Mean và Median của 2 nhóm Null và Not Null
compare_data = pd.DataFrame({
    ("Nhom Null", "Mean"): df_null_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].mean(),

    ("Nhom Null", "Median"): df_null_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].median(),

    ("Nhom Not Null", "Mean"): df_complete_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].mean(),

    ("Nhom Not Null", "Median"): df_complete_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].median()
})

display(compare_data.round(2))

Nhom Null        Nhom Not Null       
                   Mean Median          Mean Median
trip_distance     13.75   2.40          3.45   1.63
fare_amount       25.38  22.21         18.92  12.80
total_amount      31.68  28.08         28.14  21.06

In [12]:
numeric_cols=df.select_dtypes(include='number').columns
numeric_cols

Index(['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [13]:
(df[numeric_cols]<0).sum()

VendorID                     0
passenger_count              0
trip_distance                0
RatecodeID                   0
PULocationID                 0
DOLocationID                 0
payment_type                 0
fare_amount              39463
extra                    20035
mta_tax                  37730
tip_amount                  67
tolls_amount              3391
improvement_surcharge    39732
total_amount             39984
congestion_surcharge     30448
Airport_fee               9225
cbd_congestion_fee       25197
dtype: int64

In [14]:
summary= pd.DataFrame({
    "zero_count": df[numeric_cols].eq(0).sum(),
    "negative_count": df[numeric_cols].lt(0).sum()
})

display(summary)

,zero_count,negative_count
VendorID,0,0
passenger_count,14787,0
trip_distance,125738,0
RatecodeID,0,0
PULocationID,0,0
DOLocationID,0,0
payment_type,1088058,0
fare_amount,2082,39463
extra,2270914,20035
mta_tax,48423,37730


- passenger_count: Các giá trị = 0 có thể do tài xế quên nhập thủ công số khách hàng trước khi bắt đầu hành trình hoặc đây là chuyến chở hàng
- trip_distance: khoảng cách = 0 có thể do lỗi đồng hồ GPS
- fare_amount, total amount và các khoảng surchage có giá trị âm: Lệnh hoàn tiền khi tài xế bấm tiền cước sai hoặc khách hàng xảy ra tranh chấp(paymenta_type=(4,6))
- payment_type = 0 chứa 1088058 bằng chính số lượng giá trị null ở giai đoạn 1, có thể do đây là phương  thức đặt xe qua app công nghệ và ko dùng đồng hồ taxi(Kiểm tra)
- extra giá trị 0 diễn ra ngoài giờ cao điểm nên phụ phí = 0
- mta_tax=0 Do chuyến taxi chạy ngoài bang nên thuế = 0
- tolls amount: Chuyến đi cso thế không đi qua các cầu đường có hầm thu phí
- airport fee: chuyến đi ko đón khách tại sân bay
- tip_amount: Chuyến đi do khách trả bằng tiền mặt nên tiền tip ko đc tài xế nhập vô

- Các giá trị extra, mta_tax thì extra có 20000 thì trong 40 000 bị hủy, thì có mta_tax = 37 000 có thuế cũng hũy do hầu như chuyên nào cũng có nên hầu như số lượng cao hơn

In [15]:
# kiểm tra payment type = 0 có trung dữ liệu với các cột null không
print(df[df['payment_type']==0][['passenger_count','RatecodeID','store_and_fwd_flag','congestion_surcharge','Airport_fee']].isna().all())

passenger_count         True
RatecodeID              True
store_and_fwd_flag      True
congestion_surcharge    True
Airport_fee             True
dtype: bool


1.088.058 dòng NULL đồng thời ở 5 cột, trùng hoàn toàn với payment_type = 0 (Flex Fare). Đây là NULL có cấu trúc, không phải lỗi dữ liệu
- Đề xuất tạo cột đánh dấu là chuyến xe thu phí bằng Flex Fare trip

In [16]:
TotalAmount=df[df["total_amount"] < 0][
    ["fare_amount", "extra", "mta_tax", "tip_amount",
     "tolls_amount", "total_amount", "payment_type"]
]
TotalAmount
groupPayment=TotalAmount.groupby("payment_type")['payment_type'].count()
groupPayment

payment_type
0       14
1      268
2     9170
3     4493
4    26039
Name: payment_type, dtype: int64

### Đối xứng dữ liệu các dòng có total amount < 0 với các dòng có giá trị dương

In [ ]:
# 1. Lấy các cột nhận diện một chuyến
cols = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
        'PULocationID', 'DOLocationID']
# 2. Lấy 2 cột giá trị dương và giá trị âm
taNegative=df[df['total_amount']<0].copy()
taPositive=df[df['total_amount']>0].copy()
# 3. Chuyển cột giá trị âm thành giá trị dương để gộp
taNegative['tong_tien']=-taNegative['total_amount']
taPositive['tong_tien']=taPositive['total_amount']
# # 4. Tìm dòng đối xứng và gộp
taPositive['co_cap']=True

result=taNegative.merge(taPositive,on=cols + ['tong_tien'],how='left')
result['co_cap']=result['co_cap'].eq(True)

print("Tổng dòng âm:", len(result))
print("Có dòng dương đối nhau:", result['co_cap'].sum())
print("Không ghép được:", (~result['co_cap']).sum())

Tổng dòng âm: 39984
Có dòng dương đối nhau: 35707
Không ghép được: 4277


39.984 dòng có total_amount < 0, chủ yếu thuộc payment_type 4 (tranh chấp) và 2 (tiền mặt). Hệ thống giữ cả dòng gốc và thêm dòng âm để triệt tiêu,nên mỗi chuyến hủy tồn tại thành một cặp. Quyết định: giữ nguyên cả hai dòng

- Có 39 984 dong dữ liệu ghép đươc với nhau cho thấy đó không phải là lỗi
- Khoảng 4 277 dong không ghép được với nhau

In [17]:
TotalAmount=df[df["total_amount"] == 0][
    ["fare_amount", "extra", "mta_tax", "tip_amount",
     "tolls_amount", "total_amount", "payment_type"]
]
TotalAmount
groupPayment=TotalAmount.groupby("payment_type")['payment_type'].count()
groupPayment

payment_type
0     61
1     13
2     67
3    133
4    159
Name: payment_type, dtype: int64

- Giữ lại những chuyến có paymentype=(3,4) vì đó là chuyến không tính phí và tranh chấp

In [21]:
df[(df['trip_distance']==0) & (df['total_amount'].isnull())].head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


In [22]:
dur_min=(df['tpep_dropoff_datetime']-df['tpep_pickup_datetime']).dt.total_seconds()/60
dur_h=dur_min/60
speed=df['trip_distance']/dur_h.where(dur_h>0)

print("Quãng đường = 0: ",(df['trip_distance']==0).sum())
print("Quãng đường có khoảng cách lớn hơn 100km: ",(df['trip_distance'] > 100).sum())
print("Tốc độ > 80 mph:", (speed > 80).sum())
print("Tốc độ < 1 mph và thời gian > 60 phút:", ((speed < 1) & (dur_min > 60)).sum())
print(speed.quantile([.5, .95, .99, .999]).round(1))

Quãng đường = 0:  125738
Quãng đường có khoảng cách lớn hơn 100km:  162
Tốc độ > 80 mph: 874
Tốc độ < 1 mph và thời gian > 60 phút: 2565
0.500     9.3
0.950    25.0
0.990    35.2
0.999    46.8
dtype: float64


- Có khoảng 162 chuyến xe đi với khoảng cách lớn hơn 100 dặm
- Có 874 chuyến đi với tốc độ trên 80mph -> 
- Những chuyến dưới 1mph và thời gian lớn hơn 60 phút; 2565 chuyến
- 

### Kiểm tra giá trị Null cho thấy: Các chuyến đi của tập trung vào Vendor 1,2,6

- Có 1.088.058 bản ghi bị NULL đồng thời ở 5 cột passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge và Airport_fee và cũng đã kiểm chứng với phương thức payment type = 0 nên quyết đinh giữ
- Giá trị total amount = 0 chỉ giữ những chuyến có payment type = 0,3,4 vì những chuyến đặt trên app có thể đã áp dụng voucher để miến phí chuyến đi và chuyến có payment type 3,4 là chuyến đi không tính phí và tranh chấp
- Giá trị total amount < 0 xoa 4 277 chuyến không ghép cặp được
- Tốc độ > 80mph tốc độ trung vị khoảng 9.3mph, 99,9% chyến dưới 47mph nên trên 80mph có thể lỗi
- Quãng đường = 0 hoặc > 100 dặm Giờ đón, khu đón và cước vẫn đúng
- Các giá trị 0 khác: passenger_count, extra, mta_tax, tolls_amount, Airport_fee, tip_amount: Là giá trị hợp lệ theo nghiệp vụ (ngoài giờ cao điểm, không qua cầu, tiền mặt không ghi tip...)